# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Sample image unobtainable: Dryad test.zip blocked by Anubis anti-bot challenge (website route, HTTP 200 challenge page / 403) and API download requires bearer token (HTTP 401); StomaAI figshare examples return 403/202; no public mirrors. Weights verified byte-identical to the Dryad digest via the authors' HF Space copy; notebook executes setup+patch+weights verification, then stops with an explicit RuntimeError guard. No fabricated results. Resume with test.zip placed at /content/test.zip.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: StomaQuant: Deep Learning-Based Quantification for Stomatal Trait Assessment

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-3146fc78c8a19cb00186903444524a1b)


# StomaQuant — Minimal reproduction: YOLOv12 stomatal detection and trait inference

**Paper:** Lau K.X. et al., *StomaQuant: Deep Learning-Based Quantification for Stomatal Trait Assessment*, bioRxiv 2026.01.06.698068v1 (posted 2026-01-07). DOI: [10.64898/2026.01.06.698068](https://doi.org/10.64898/2026.01.06.698068)
**Author code:** https://github.com/kjxlau/StomaQuant (training scripts, AGPL-3.0)
**Released inference tool:** Hugging Face Space [TemasekLifesciLab/StomaQuant](https://huggingface.co/spaces/TemasekLifesciLab/StomaQuant) (`app.py`) — the paper's "Image analysis and stomatal traits inference" implementation.
**Data/weights:** Dryad doi:[10.5061/dryad.3j9kd51zr](https://datadryad.org/dataset/doi:10.5061/dryad.3j9kd51zr), version 420635.

## ⚠️ BLOCKED INPUT — NOT A VALIDATED REPRODUCTION / ⚠️ 入力データが取得できず未検証

**Status: this notebook is BLOCKED at the sample-image acquisition step and is NOT a validated reproduction.** The released YOLOv12 checkpoint is downloaded and SHA-256-verified here (byte-identical to the Dryad deposit), and the environment/patch/model-loading path was verified — but **no sample image could be acquired through authorized public routes** (details and observed errors in the acquisition cell and its output). A deliberate `RuntimeError` guard stops execution there so no misleading result can be produced. The trait-analysis cells are kept as the faithful, unexecuted procedure.

* As of 2026-10-08 the Dryad website download route is protected by an anti-bot challenge (Anubis) and the Dryad API download endpoint requires an authenticated token — automated acquisition of `test.zip` is not possible without circumventing access controls, which this notebook does not do. A learner with a normal browser can still download the dataset manually and adapt the acquisition cell.
* **日本語:** このノートブックはサンプル画像の取得段階で **ブロック** されており、検証済みの再現ではありません。重みは著者の公開ミラーから取得し、Dryad記録のSHA-256と完全一致することを確認済みです。一方、サンプル画像（Dryad `test.zip`）は2026-10-08時点でアンチボット保護（Anubis）やAPI認証により自動取得できず、アクセス制御を回避しないため実行を停止します。ブラウザで手動ダウンロードしたファイルを使えば解析セルはそのまま動作します。

**AI-assisted glue notice:** The scientific operations below (pinned model-utility clone, attention patch, model load, prediction, trait formulas) are copied or adapted from the authors' released `app.py`. Sample selection, download verification, and reference-annotation overlay are AI-generated glue and are labeled as such. The authors did not provide this notebook.
**日本語:** 科学的処理は著者の `app.py` からコピー／翻案しています。ダウンロード検証・サンプル選択・参照アノテーション描画はAI生成の補助コードです。

## Runtime selection — CPU (faithful to the released tool)

The released StomaQuant Space pins `CUDA_VISIBLE_DEVICES=""`, i.e. it **runs on CPU by design**. Inference on a single 1392×1040 microscope image with YOLOv12s takes seconds to about a minute on CPU. **No GPU is needed**: select *Runtime → Change runtime type → CPU*.
**日本語:** 公開ツール自体がCPU動作のため、CPUランタイムで十分です（GPU不要）。セットアップ+検証は数分で終わります。

### 1. Clone the authors' pinned YOLOv12 utility repository

The custom YOLOv12 checkpoints (A3-attention architecture) require the `ultralytics` fork from `sunsmarterjie/yolov12`. The released `app.py` clones this exact commit `3a336a4adf3683d280e1a50d03fa24bbe7f24a5b`, so we do the same (detached HEAD at the pinned commit, verified).
**日本語:** 著者の `app.py` と同じく、`sunsmarterjie/yolov12` を指定コミットでクローンします。

In [ ]:
import subprocess, pathlib

REPO_URL = "https://github.com/sunsmarterjie/yolov12.git"
PINNED_COMMIT = "3a336a4adf3683d280e1a50d03fa24bbe7f24a5b"
REPO_DIR = pathlib.Path("/content/yolov12")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True, timeout=300)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", PINNED_COMMIT], check=True)
head = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                      check=True, capture_output=True, text=True).stdout.strip()
assert head == PINNED_COMMIT, f"checkout mismatch: {head}"
print("yolov12 pinned at", head)

### 2. Apply the authors' "Hybrid Patch" to the attention module

`app.py` patches `ultralytics/nn/modules/block.py` so the A3 attention layer works with **both** checkpoint styles (fused `qkv` and separate `qk`/`v`). The regex patterns and replacement logic below are copied from the authors' released `app.py` (Step 2: "Apply Hybrid Patch"). This is required to load `yolov12_300epochs.pt`.
**日本語:** 著者の `app.py` に含まれる「Hybrid Patch」をそのまま適用します。

In [ ]:
import re

block_py = REPO_DIR / "ultralytics" / "nn" / "modules" / "block.py"
content = block_py.read_text()

hybrid_logic = """
        if hasattr(self, 'qkv'):
            qkv = self.qkv(x).flatten(2).transpose(1, 2)
        else:
            qk = self.qk(x)
            v = self.v(x)
            qkv = torch.cat((qk, v), dim=1).flatten(2).transpose(1, 2)
        """

pattern_original = r"qkv\s*=\s*self\.qkv\(x\)\.flatten\(2\)\.transpose\(1,\s*2\)"
pattern_seg_only = (r"qk\s*=\s*self\.qk\(x\).*?v\s*=\s*self\.v\(x\).*?"
                    r"qkv\s*=\s*torch\.cat\(\(qk,\s*v\),\s*dim=1\)\.flatten\(2\)\.transpose\(1,\s*2\)")

if "if hasattr(self, 'qkv'):" in content:
    print("Hybrid patch already applied.")
elif re.search(pattern_seg_only, content, flags=re.DOTALL):
    content = re.sub(pattern_seg_only, hybrid_logic, content, flags=re.DOTALL)
    block_py.write_text(content); print("Upgraded seg-only patch to hybrid.")
elif re.search(pattern_original, content):
    content = re.sub(pattern_original, hybrid_logic, content)
    block_py.write_text(content); print("Hybrid patch applied.")
else:
    raise RuntimeError("No recognizable A3 attention pattern in block.py; app.py patch would fail.")

if "import torch" not in content:
    block_py.write_text("import torch\n" + block_py.read_text())
print("block.py patched OK")

### 3. Install the one missing fork dependency and import through the pinned fork

The fork imports `thop` at module level (its own `ultralytics/utils/torch_utils.py`); Colab does not preinstall it, so we install it explicitly. Then, exactly as `app.py` does (`sys.path.insert(0, ...)`), the local patched fork shadows any preinstalled `ultralytics` and `YOLO` is imported from it.
**日本語:** フォークが必要とする `thop` を明示的にインストールし、パッチ済みフォークを優先して `YOLO` をimportします。

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "thop"], check=True)

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR.resolve()))  # same as app.py's second insert

import torch
from ultralytics import YOLO

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
import ultralytics
print("ultralytics from:", ultralytics.__file__)

### 4. Download and SHA-256-verify the released YOLOv12 checkpoint

**AI-generated glue cell.** The paper's declared source is Dryad file `YOLOv12_trained_weights.pt` (18,758,947 B, sha256 `562e4851…9500`, recorded from the Dryad API file listing of version 420635). The Dryad API download endpoint returned HTTP 401 ("must have current bearer token") for anonymous requests, so we use the authors' own public copy on the StomaQuant Space (`yolov12_300epochs.pt`) and **verify its SHA-256 against the Dryad-recorded digest** — proving byte identity with the Dryad deposit without relying on the blocked route. All bytes stay inside the Colab VM.
**日本語:** 重みはDryad記録のSHA-256と一致することを確認した上で、著者自身の公開ミラー（StomaQuant Space）から取得します。バイト単位でDryad収録物と同一であることを検証済みです。

In [ ]:
import hashlib, pathlib, requests

WEIGHTS_SHA256 = "562e4851c32813f204e97c92ef6c741f1d0c5773954db98e24937e1e81919500"
WEIGHTS_SIZE = 18758947  # bytes, from Dryad version 420635 file listing (2026-10-08)
WEIGHTS_URL = "https://huggingface.co/spaces/TemasekLifesciLab/StomaQuant/resolve/main/yolov12_300epochs.pt"

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

dest = pathlib.Path("/content/YOLOv12_trained_weights.pt")
if not dest.exists():
    with requests.get(WEIGHTS_URL, timeout=600, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
assert dest.stat().st_size == WEIGHTS_SIZE, f"size {dest.stat().st_size} != {WEIGHTS_SIZE}"
got = sha256_of(dest)
assert got == WEIGHTS_SHA256, f"sha256 {got} != recorded {WEIGHTS_SHA256}"
print(f"YOLOv12_trained_weights.pt: {WEIGHTS_SIZE} bytes, SHA-256 verified (identical to Dryad deposit)")

### 5. Acquire a sample image — **BLOCKED (documented failure)**

**AI-generated glue cell.** The minimal reproduction needs one test image from the Dryad deposit (`test.zip`, file id 4566273, sha256 `8044118c…1009`). Two authorized public routes were attempted (observed 2026-10-08, from this runtime and from a host client):

1. Dryad website route `https://datadryad.org/downloads/file_stream/4566273` → HTTP 403 for plain clients; HTTP 200 but an **Anubis anti-bot proof-of-work challenge page** for browser-like clients (title "Validating..."). Solving it programmatically would circumvent an access control, which this notebook refuses to do.
2. Dryad API route `https://datadryad.org/api/v2/files/4566273/download` → HTTP 401 `{"error":"Unauthorized, must have current bearer token"}`.

The paper's other cited image source (StomaAI examples on `adelaide.figshare.com`, used for the paper's unseen-data testing) also returns HTTP 403/202 to automated clients. **No authorized public route could yield the sample bytes, so execution stops here with an explicit error rather than substituting or fabricating data.** A learner who downloads `test.zip` manually (normal browser) can place it at `/content/test.zip` and re-run from the next cell.
**日本語:** Dryadのウェブ経路はアンチボット保護（Anubis）により、API経路は認証トークン要求により、自動取得できませんでした（2026-10-08観測）。データを捏造・代替しないため、ここで明示的にエラーで停止します。ブラウザで `test.zip` を手動取得して `/content/test.zip` に置けば、以降のセルはそのまま実行できます。

In [ ]:
import pathlib, zipfile, requests

SAMPLE_ACQUIRED = False
SAMPLE_IMG = None
SAMPLE_LABEL = None

TEST_ZIP = pathlib.Path("/content/test.zip")
if TEST_ZIP.exists():
    SAMPLE_ACQUIRED = True
else:
    ATTEMPTS = [
        ("Dryad website file_stream",
         "https://datadryad.org/downloads/file_stream/4566273", {}),
        ("Dryad API stash:download",
         "https://datadryad.org/api/v2/files/4566273/download", {}),
    ]
    observed = []
    for name, url, headers in ATTEMPTS:
        try:
            r = requests.get(url, headers=headers, timeout=120, stream=True)
            ctype = r.headers.get("Content-Type", "")
            observed.append(f"{name}: HTTP {r.status_code} ({ctype})")
            r.close()
        except Exception as e:
            observed.append(f"{name}: {type(e).__name__}: {str(e)[:120]}")
    print("attempted routes ->", "; ".join(observed))
    print("Dryad test.zip (file 4566273) is not obtainable through authorized public routes.")
    raise RuntimeError(
        "BLOCKED: no authorized public route can download the Dryad sample (test.zip, id 4566273). "
        "Website route is behind an Anubis anti-bot challenge; API download requires a bearer token. "
        "Manually place test.zip at /content/test.zip to continue with the real paper data. "
        "No substitute or synthetic input is used.")

### 6. Extract `test.zip` and deterministically select one sample *(runs only if the sample was acquired)*

**AI-generated glue cell.** Extracts the archive under `/content` and picks the **lexicographically first image** (with its YOLO-format `.txt` label when present) as the single minimal example. The deterministic rule avoids cherry-picking.
**日本語:** （サンプル取得済みの場合のみ実行）`test.zip` を展開し、ファイル名順で最初の画像とそのYOLOラベルを決定論的に選びます。

In [ ]:
if not SAMPLE_ACQUIRED:
    raise RuntimeError("StomaQuant minimal reproduction is BLOCKED: no sample image could be acquired through authorized public routes (see the sample-acquisition cell and report). No result is produced; nothing here is fabricated.")

import zipfile, cv2

EXTRACT_DIR = pathlib.Path("/content/stomaquant_test")
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile("/content/test.zip") as zf:
        zf.extractall(EXTRACT_DIR)
with zipfile.ZipFile("/content/test.zip") as zf:
    names = zf.namelist()

img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
images = sorted(p for p in EXTRACT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in img_exts)
assert images, "no image found in test.zip"
SAMPLE_IMG = images[0]
SAMPLE_LABEL = SAMPLE_IMG.with_suffix(".txt")
SAMPLE_LABEL = SAMPLE_LABEL if SAMPLE_LABEL.exists() else None
print("entries:", len(names), "| images found:", len(images))
print("selected sample:", SAMPLE_IMG)
print("reference label:", SAMPLE_LABEL)
_img = cv2.imread(str(SAMPLE_IMG))
print("image size (WxH):", _img.shape[1], "x", _img.shape[0])

### 7. Input preview with reference annotation (before inference) *(guarded)*

**AI-generated glue cell.** Shows the selected microscopy image with its **reference (ground-truth) YOLO annotation** from the Dryad deposit, if a label file exists. Class ids appear as `GT <id>`; the id→name mapping is confirmed against `model.names` after loading (the paper's labelme workflow used classes *stomata* and *pore*).
**日本語:** （ガード付き）選択画像とDryadの参照アノテーションを表示します。クラス名はモデル読み込み後に `model.names` で確認します。

In [ ]:
if not SAMPLE_ACQUIRED:
    raise RuntimeError("StomaQuant minimal reproduction is BLOCKED: no sample image could be acquired through authorized public routes (see the sample-acquisition cell and report). No result is produced; nothing here is fabricated.")

import cv2
import matplotlib.pyplot as plt

bgr = cv2.imread(str(SAMPLE_IMG))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
H, W = rgb.shape[:2]
preview = rgb.copy()

if SAMPLE_LABEL is not None:
    for line in SAMPLE_LABEL.read_text().strip().splitlines():
        parts = line.split()
        cid = int(parts[0]); xc, yc, w, h = map(float, parts[1:5])
        x1, y1 = int((xc - w / 2) * W), int((yc - h / 2) * H)
        x2, y2 = int((xc + w / 2) * W), int((yc + h / 2) * H)
        cv2.rectangle(preview, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(preview, f"GT {cid}", (x1, max(12, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1)

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(preview)
ax.set_title(f"Input: {SAMPLE_IMG.name} ({W}x{H}px)\nblue = reference YOLO annotation from Dryad"
             + (" (none in deposit)" if SAMPLE_LABEL is None else ""))
ax.axis("off")
plt.tight_layout(); plt.savefig("/content/preview_reference.png", dpi=110); plt.show()
print("GT boxes drawn from:", SAMPLE_LABEL if SAMPLE_LABEL is not None else "no label file")

### 8. Load the released YOLOv12 checkpoint (author operation)

Loads the SHA-256-verified `YOLOv12_trained_weights.pt` through the patched fork exactly as `app.py` does (`YOLO(path)`), and prints `model.names` so the class mapping used by the reference annotation is confirmed. No weights are invented or substituted.
**日本語:** 検証済みの重みを `app.py` と同じ方法で読み込み、`model.names` でクラス対応を確認します。

In [ ]:
if not SAMPLE_ACQUIRED:
    raise RuntimeError("StomaQuant minimal reproduction is BLOCKED: no sample image could be acquired through authorized public routes (see the sample-acquisition cell and report). No result is produced; nothing here is fabricated.")

WEIGHTS = "/content/YOLOv12_trained_weights.pt"
model = YOLO(WEIGHTS)
print("Model loaded. class names:", model.names)

### 9. Run inference on the sample (author operation, CPU) *(guarded)*

`model.predict(..., conf=0.5, retina_masks=True)` with **CPU** device — the exact call and default confidence threshold (0.5) of the released Space tool, which pins CPU.
**日本語:** （ガード付き）公開ツールと同じ `conf=0.5`・`retina_masks=True`、`device="cpu"` で推論します。

In [ ]:
if not SAMPLE_ACQUIRED:
    raise RuntimeError("StomaQuant minimal reproduction is BLOCKED: no sample image could be acquired through authorized public routes (see the sample-acquisition cell and report). No result is produced; nothing here is fabricated.")

CONF_THRESHOLD = 0.5  # released Space default (gr.Slider(0.0, 1.0, 0.5, ...))

results = model.predict(str(SAMPLE_IMG), conf=CONF_THRESHOLD, device="cpu",
                        retina_masks=True, verbose=False)
result = results[0]
print("detections:", len(result.boxes))
print("image shape used by model:", result.orig_shape)

### 10. Compute stomatal traits with the authors' metric functions (verbatim, guarded)

`calculate_class_metrics` and `calculate_gmax` below are **copied verbatim** from the released `app.py` (lines 171–210). For the detection model, per-object pixel area is the bounding-box area (the Space's own fallback for mask-less models). Calibration: the paper states the BX53 scale is **0.32 µm/pixel at 20×** and **0.16 µm/pixel at 40×**; the mapping is set from the sample's filename when it encodes crop/magnification (author deposit files do), otherwise 0.32 µm/px is used with a printed note.
**日本語:** （ガード付き）面積・密度・gmaxの計算関数は公開 `app.py` からの逐語コピーです。較正値は論文記載のBX53スケールに基づきます。

In [ ]:
if not SAMPLE_ACQUIRED:
    raise RuntimeError("StomaQuant minimal reproduction is BLOCKED: no sample image could be acquired through authorized public routes (see the sample-acquisition cell and report). No result is produced; nothing here is fabricated.")

import cv2

def calculate_class_metrics(areas_pixels, total_image_pixels, microns_per_pixel, class_name):
    # verbatim from TemasekLifesciLab/StomaQuant app.py (released tool)
    count = len(areas_pixels)
    sq_microns_per_pixel = microns_per_pixel ** 2
    image_area_mm2 = (total_image_pixels * sq_microns_per_pixel) / 1_000_000
    if count == 0:
        return {
            f"{class_name} Count": 0,
            f"{class_name} Total Area (sq µm)": 0.0,
            f"{class_name} Avg Area (sq µm)": 0.0,
            f"{class_name} Density (per mm²)": 0.0
        }, 0.0
    total_object_area_pixels = sum(areas_pixels)
    total_object_area_microns = total_object_area_pixels * sq_microns_per_pixel
    avg_object_area_microns = total_object_area_microns / count
    density_per_mm2 = count / image_area_mm2 if image_area_mm2 > 0 else 0.0
    metrics = {
        f"{class_name} Count": count,
        f"{class_name} Total Area (sq µm)": round(total_object_area_microns, 2),
        f"{class_name} Avg Area (sq µm)": round(avg_object_area_microns, 2),
        f"{class_name} Density (per mm²)": round(density_per_mm2, 2)
    }
    return metrics, avg_object_area_microns

def calculate_gmax(d_diffusivity, pore_depth_um, stomata_count, avg_pore_area_um2, image_area_mm2):
    # verbatim from TemasekLifesciLab/StomaQuant app.py (released tool)
    if pore_depth_um <= 0 or image_area_mm2 <= 0 or stomata_count == 0: return 0.0
    p_density_m2 = (stomata_count / image_area_mm2) * 1_000_000
    D_area_m2 = avg_pore_area_um2 * 1e-12
    l_depth_m = pore_depth_um * 1e-6
    gmax_m_s = (d_diffusivity * p_density_m2 * D_area_m2) / l_depth_m
    return gmax_m_s

name_upper = SAMPLE_IMG.name.upper()
if "RICE" in name_upper or "40X" in name_upper.replace("×", "X"):
    MICRONS_PER_PIXEL = 0.16  # paper: BX53 40x objective
    calib_source = "0.16 um/px (paper, 40x) — from filename"
else:
    MICRONS_PER_PIXEL = 0.32  # paper: BX53 20x objective
    calib_source = "0.32 um/px (paper, 20x) — default for barley/sugarcane"
print("Calibration:", calib_source, "| sample:", SAMPLE_IMG.name)

total_image_pixels = H * W
image_area_mm2 = (total_image_pixels * MICRONS_PER_PIXEL ** 2) / 1_000_000

stomata_areas, pore_areas = [], []
cls_ids = result.boxes.cls.cpu().numpy().astype(int)
names = result.names
boxes = result.boxes.xyxy.cpu().numpy()
for i, class_id in enumerate(cls_ids):
    cname = names[class_id].lower()
    area = (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1])  # Space fallback for box models
    if "pore" in cname: pore_areas.append(area)
    elif "stoma" in cname: stomata_areas.append(area)

stomata_metrics, avg_stomata_area = calculate_class_metrics(
    stomata_areas, total_image_pixels, MICRONS_PER_PIXEL, "Stomata")
pore_metrics, avg_pore_area_um = calculate_class_metrics(
    pore_areas, total_image_pixels, MICRONS_PER_PIXEL, "Pore")

# gmax inputs: Space defaults d=2.49e-5 m^2/s, pore depth l=10 um
GMAX_D, PORE_DEPTH_UM = 2.49e-5, 10.0
gmax_val = calculate_gmax(GMAX_D, PORE_DEPTH_UM, stomata_metrics["Stomata Count"],
                          avg_pore_area_um, image_area_mm2)

print(f"image area: {total_image_pixels} px = {image_area_mm2:.4f} mm^2")
print("Stomata:", stomata_metrics)
print("Pore:", pore_metrics)
print(f"gmax = {gmax_val:.6f} m/s  (d={GMAX_D}, l={PORE_DEPTH_UM} um; 0 if no pore detections)")

### 11. The paper's pixel-fraction density formula (comparison row, guarded)

The paper states `Stomatal Density = (stomata pixels count ÷ total image pixels) × 100`, while the released tool computes count per mm². We compute the paper's formula for the same detections using the predicted box areas as the "stomata pixels" proxy (box area is the detection model's own pixel measure in the Space), and show both clearly labeled — this makes the flow's flagged discrepancy transparent.
**日本語:** （ガード付き）論文式（気孔ピクセル数÷全ピクセル×100）と公開ツール式（count/mm²）は定義が異なります。両方を明示して比較します。

In [ ]:
if not SAMPLE_ACQUIRED:
    raise RuntimeError("StomaQuant minimal reproduction is BLOCKED: no sample image could be acquired through authorized public routes (see the sample-acquisition cell and report). No result is produced; nothing here is fabricated.")

stomata_pixels = sum(stomata_areas)
paper_density_pct = (stomata_pixels / total_image_pixels) * 100 if total_image_pixels else 0.0
print(f"Paper formula: stomatal density = {paper_density_pct:.3f} % of image area (box-area proxy)")
print(f"Released tool: stomatal density = {stomata_metrics['Stomata Density (per mm²)']} per mm^2")

### 12. Prediction overlay and results table (CSV export, guarded)

Left: input with reference annotation. Right: the model's own annotated rendering from `result.plot()` (what the Space writes out) with per-detection labels. The trait table is exported to `/content/stomaquant_results.csv`.
**日本語:** （ガード付き）参照アノテーションとモデル予測を並べて表示し、結果表をCSVに保存します。

In [ ]:
if not SAMPLE_ACQUIRED:
    raise RuntimeError("StomaQuant minimal reproduction is BLOCKED: no sample image could be acquired through authorized public routes (see the sample-acquisition cell and report). No result is produced; nothing here is fabricated.")

import pandas as pd

annotated_bgr = result.plot()  # ultralytics renderer, same as Space's annotated output
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(preview)
axes[0].set_title(f"Reference (Dryad YOLO label): {len(SAMPLE_LABEL.read_text().strip().splitlines()) if SAMPLE_LABEL is not None else 0} boxes")
axes[1].imshow(annotated_rgb)
axes[1].set_title(f"YOLOv12 prediction: {len(result.boxes)} detections (conf>={CONF_THRESHOLD}, CPU)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig("/content/reference_vs_prediction.png", dpi=110); plt.show()

summary = {
    "Filename": SAMPLE_IMG.name, "Model": "YOLOv12", "Runtime": "CPU",
    "MicronsPerPixel": MICRONS_PER_PIXEL, "ConfThreshold": CONF_THRESHOLD,
    "gmax (m/s)": f"{gmax_val:.6f}", **stomata_metrics, **pore_metrics,
    "Paper density (stomata px % of image)": round(paper_density_pct, 3),
}
df = pd.DataFrame([summary])
df.to_csv("/content/stomaquant_results.csv", index=False)
print(df.T.to_string(header=False))

## Interpretation, blockers, and validation record

- **Verified in this run (executed outputs above):** the authors' pinned `yolov12` fork clones and applies the released "Hybrid Patch"; the released YOLOv12 checkpoint downloads and its **SHA-256 matches the Dryad-recorded digest exactly** — the Space copy is byte-identical to the paper's deposit.
- **Blocked (documented, not invented):** the sample image. Dryad's website route is behind an Anubis proof-of-work anti-bot challenge and its API download requires a bearer token (both observed 2026-10-08); the paper's other cited image source (StomaAI examples on `adelaide.figshare.com`) also rejects automated clients (403/202). No authorized public route could yield the sample bytes, so inference and trait computation **did not run** and no results are claimed.
- **How to unblock:** a learner with a normal browser can download `test.zip` from the Dryad dataset page and place it at `/content/test.zip`; every subsequent cell then runs as written (the trait functions are the released tool's, verbatim).
- **What a successful run would show:** the paper's Fig. 5/10/11 pipeline on one example — stomata count, size (µm²), density (per mm² and the paper's pixel-fraction %), and the optional gmax estimate. A one-image run would still not verify dataset-level results (mAP, per-variety comparisons).
- **Calibration (from the paper):** BX53 20× → 0.32 µm/px, 40× → 0.16 µm/px; images 1392×1040 px.
- **Provenance:** weights `yolov12_300epochs.pt` = Dryad `YOLOv12_trained_weights.pt` (sha256 verified); model-utility code `sunsmarterjie/yolov12` @ `3a336a4a`; trait functions verbatim from the released Space `app.py` (repository code AGPL-3.0; Space code has no license file — used here under fair scholarly-reuse citation, not redistributed).

**日本語:** 本実行で検証できたのは環境構築・パッチ適用・重みのSHA-256一致までです。サンプル画像は両リポジトリのアクセス制御により取得できず、推論・形質計算は**未実行**であり、結果は一切捏造していません。ブラウザで `test.zip` を取得して `/content/test.zip` に置けば、以降のセルはそのまま動作します。

### References
1. Lau et al., bioRxiv 2026.01.06.698068v1 — https://doi.org/10.64898/2026.01.06.698068
2. Author repository — https://github.com/kjxlau/StomaQuant (commit `7224dcb5`)
3. Released tool (inference + traits) — https://huggingface.co/spaces/TemasekLifesciLab/StomaQuant (`app.py`)
4. YOLOv12 fork — https://github.com/sunsmarterjie/yolov12 @ `3a336a4a` (via `app.py`)
5. Data/weights — Dryad doi:10.5061/dryad.3j9kd51zr, version 420635